> **Rama:** `feature/preprocessing`
>
> Carga `X`, `y`, `numeric_all` y `categorical_all` desde `src/artifacts/02_feature_eng.joblib` (generado por `feautre/feature-eng`). Al final guarda `X_train`, `X_test`, `y_train`, `y_test`, `preprocessor` y las listas de columnas en `src/artifacts/03_preprocessing.joblib` para que la rama `feature/modeling` los cargue.

In [1]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

RANDOM_STATE = 42

_art = joblib.load('src/artifacts/02_feature_eng.joblib')
X = _art['X']
y = _art['y']
numeric_all = _art['numeric_all']
categorical_all = _art['categorical_all']
print(f'X: {X.shape}  |  y: {y.shape}')

X: (4424, 20)  |  y: (4424,)


## **Paso 5: División train / test**

Separamos un 20% para test, estratificando por `Target` para conservar las proporciones de clase en
ambos conjuntos. El conjunto de test no se vuelve a tocar hasta la evaluación final (Paso 10).

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} filas   Test: {X_test.shape[0]} filas')
print('\nProporción de clases en train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporción de clases en test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 3539 filas   Test: 885 filas

Proporción de clases en train:
Target
Graduate    0.499
Dropout     0.321
Enrolled    0.179
Name: proportion, dtype: float64

Proporción de clases en test:
Target
Graduate    0.499
Dropout     0.321
Enrolled    0.180
Name: proportion, dtype: float64


## **Paso 6: Preprocesamiento**

Dentro de las variables categóricas seleccionadas distinguimos dos grupos, para no aplicar
one-hot innecesariamente:

- **Binarias (ya en 0/1):** `Displaced`, `Debtor`, `Tuition fees up to date`, `Gender`,
  `Scholarship holder` → se dejan tal cual (`passthrough`).
- **Nominales con varias categorías:** `Marital status`, `Application mode`, `Application order`,
  `Course`, `Daytime/evening attendance`, `Previous qualification`, cualificaciones y ocupaciones de
  los padres → `OneHotEncoder` (con `handle_unknown='ignore'` para robustez ante categorías no vistas
  en train).
- **Numéricas:** `Previous qualification (grade)`, `Admission grade`, `Age at enrollment`,
  `Unemployment rate`, `GDP` → `StandardScaler`, necesario para los modelos lineales y de margen
  (Logistic Regression, SVM).

Todo el preprocesamiento se encapsula en un `ColumnTransformer` dentro de un `Pipeline`, para que el
mismo objeto se pueda usar en validación cruzada, búsqueda de hiperparámetros y, finalmente, en
producción sin fugas de información entre folds.

In [3]:
binary_cols = ['Displaced', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder']
nominal_cols = [c for c in categorical_all if c in X.columns and c not in binary_cols]
numeric_cols = [c for c in numeric_all if c in X.columns]

print(f'Numéricas ({len(numeric_cols)}): {numeric_cols}')
print(f'Nominales ({len(nominal_cols)}): {nominal_cols}')
print(f'Binarias  ({len(binary_cols)}): {binary_cols}')

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols),
    ('bin', 'passthrough', binary_cols)
])

Numéricas (5): ['Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'Unemployment rate', 'GDP']
Nominales (10): ['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation"]
Binarias  (5): ['Displaced', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder']


### Guardado de artefactos para la siguiente rama (`feature/modeling`)

In [4]:
import os
os.makedirs('src/artifacts', exist_ok=True)
joblib.dump({
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
    'preprocessor': preprocessor,
    'numeric_cols': numeric_cols, 'nominal_cols': nominal_cols, 'binary_cols': binary_cols,
    'RANDOM_STATE': RANDOM_STATE,
}, 'src/artifacts/03_preprocessing.joblib')
print('Artefactos guardados en src/artifacts/03_preprocessing.joblib')

Artefactos guardados en src/artifacts/03_preprocessing.joblib
